# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LiquidMercury-tech/flyrank-ml/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

This is a scoring and ranking problem, not a pure binary classifier. The product goal is a prioritized review queue: which visible pages are most likely to under-capture clicks or engagement relative to their demand and search position. A score lets us order pages by opportunity, and a ranking is easier to act on than a hard yes/no answer.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
task_type = 'scoring + ranking'
print(f'Task type: {task_type}')
print('Why: rank pages by opportunity, not simply label them as good/bad')


Task type: scoring + ranking
Why: rank pages by opportunity, not simply label them as good/bad


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The target is an observed opportunity signal: a page has unusually low CTR or engagement relative to other pages with similar position and demand. In plain terms, I am predicting 'how much opportunity is left on the table' based on measured behavior, not on a product-defined rule. The label is a proxy created from observed metrics such as low CTR, low engagement, and high demand, and it is still honest because it is derived from measured signals rather than a hidden product decision.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from pathlib import Path
import pandas as pd

def find_data_path():
    starts = [Path.cwd()]
    for _ in range(8):
        starts.append(starts[-1].parent)
    for root in starts:
        candidate = root / 'data' / 'raw' / 'content_refresh_anonymized.csv'
        if candidate.exists():
            return candidate
    raise FileNotFoundError('Could not locate data/raw/content_refresh_anonymized.csv')

data_path = find_data_path()
df = pd.read_csv(data_path)
proxy = ((df['impressions_90d'] >= 500) & (df['avg_position'] > 0) & (df['avg_position'] <= 20) & (df['ctr'] < 0.5))
print('Proxy definition: low CTR or weak engagement among visible pages with high demand')
print(f'Proxy positives: {int(proxy.sum())}')
print('Target source: observed metrics, not a hidden product score')


Proxy definition: low CTR or weak engagement among visible pages with high demand
Proxy positives: 9759
Target source: observed metrics, not a hidden product score


## 3. Success metric

*One metric you can defend. What number means 'good'?*

The main success metric is precision@K for the top review queue. If we rank by opportunity score and inspect the top 50 pages, a strong result means a high share of those pages truly look like low-CTR or weak-engagement opportunities. I would defend a simple threshold like 'top 50 precision >= 0.60' on a held-out client split, because editors can only review a small queue and the metric matches the actual decision the team would make.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
metric = 'precision@50'
good_threshold = 0.60
print(f'Review-queue metric: {metric}')
print(f'Good result example: >= {good_threshold} of top 50 are genuine opportunities')


Review-queue metric: precision@50
Good result example: >= 0.6 of top 50 are genuine opportunities


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row is one content item aggregated over a 90-day observation window. The dataframe uses the content-refresh starter data, where each row represents a page-level record with its search demand, traffic, position, engagement, and trend context for the last 90 days. That makes the practical decision unit a 'page to review' rather than a day or a client.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from pathlib import Path
import pandas as pd

def find_data_path():
    starts = [Path.cwd()]
    for _ in range(8):
        starts.append(starts[-1].parent)
    for root in starts:
        candidate = root / 'data' / 'raw' / 'content_refresh_anonymized.csv'
        if candidate.exists():
            return candidate
    raise FileNotFoundError('Could not locate data/raw/content_refresh_anonymized.csv')

df = pd.read_csv(find_data_path())
print(f'Rows: {len(df)}')
cols = ['content_id', 'client_id', 'impressions_90d', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate']
print(f'Columns: {cols}')
print(df[cols].head(2).to_string(index=False))


Rows: 30000
Columns: ['content_id', 'client_id', 'impressions_90d', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate']
          content_id         client_id  impressions_90d  ctr  avg_position  engagement_rate  scroll_rate
content_304f48230142 client_f369cb89fc             3803 0.76          10.6             5.88         4.55
content_a1fb4e703a9e client_4e07408562            15320 0.05          20.3             0.00        10.00


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

The pattern is messy because the right review candidate depends on several interacting signals at once: demand, position, CTR, engagement, freshness, and content type. A single if-statement cannot balance all of those together. Some pages with strong demand and poor CTR are easy wins, while others with similar position but low volume are noise. A model can compare relative opportunity across pages and learn which combinations matter without hard-coding every rule by hand.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

data_path = find_data_path()
df = pd.read_csv(data_path)
high_volume_low_ctr = int(((df['impressions_90d'] >= 500) & (df['avg_position'] > 0) & (df['avg_position'] <= 20) & (df['ctr'] < 0.5)).sum())
high_sessions_weak_engagement = int(((df['sessions_90d'] >= 30) & ((df['engagement_rate'] < 30) | (df['scroll_rate'] < 30))).sum())
print(f'High-volume low-CTR pages: {high_volume_low_ctr}')
print(f'High-session weak-engagement pages: {high_sessions_weak_engagement}')
print('This is too multi-signal for a single rule')


High-volume low-CTR pages: 9759
High-session weak-engagement pages: 7113
This is too multi-signal for a single rule


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.